In [1]:
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
import requests
from datetime import datetime
import re
from time import sleep

In [3]:
initial_url = "https://crossroadsleague.com/sports/bsb/2006-07/schedule?jsRendering=true"

In [4]:
headers = {
    "User-Agent": "Chrome/120.0.0.0",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://crossroadsleague.com/"
}

page = requests.get(initial_url, headers=headers).text
soup = BeautifulSoup(page, 'html.parser')

In [5]:
column_headers = ['date', 'away', 'home', 'away_score', 'home_score']
df = pd.DataFrame(columns=column_headers)

# Scraping Data Across Every Game

In [6]:
urls = [option.get('value') for option in soup.find('select', id='season-selector').find_all('option')]

In [7]:
def scrape(url):

    page = requests.get("https://crossroadsleague.com" + url, headers=headers).text
    
    soup = BeautifulSoup(page, 'html.parser')
    sleep(5)
    print("Season Result: ", soup.find('div', class_='page-content-header').text.strip().split(' ')[0])
    season = soup.find('div', class_='page-content-header').text.strip().split(' ')[0]
    
    start_year = None
    end_year = None
    
    if '-' in season:
        #start_year = int(season.split('-')[0])
        start_year = int(re.split(r'-', season)[0])
    else:
        end_year = int(season)

    # Finding the score results
    results = soup.find_all('div', class_='result')
    
    for result in results:

        # Date
        day = result.find_parent('div', class_='section-event-date').get('data-date')
        month = result.find_parent('div', class_='section-event-month').get('class')[1]

        month_num = datetime.strptime(month, "%B").month

        if month_num >= 8 and start_year:
            year = start_year
        elif month_num >= 8 and end_year:
            year = end_year - 1
            
        if month_num < 8 and start_year:
            year = start_year + 1
        elif month_num < 8 and end_year:
            year = end_year

        date_str = f"{month} {day} {year}"
        date = datetime.strptime(date_str, "%B %a. %d %Y")
        
        # Team Name
        team_name = result.find_all('span', class_='team-name')[0].text.strip()
        
        # Opponent Name
        opponent_name = result.find_all('span', class_='team-name')[1].text.strip()

        # Scores
        team_score = result.find_all('div', class_='flex-shrink-1')[0].text.strip()
        opponent_score = result.find_all('div', class_='flex-shrink-1')[1].text.strip()
        
        df.loc[len(df)] = [date, team_name, opponent_name, team_score, opponent_score]

In [8]:
for url in urls:
    scrape(url)
df

Season Result:  2026
Season Result:  2025
Season Result:  2024
Season Result:  2022-23
Season Result:  2021-22
Season Result:  2020-21
Season Result:  2019-20
Season Result:  2018-19
Season Result:  2017-18
Season Result:  2016-17
Season Result:  2015-16
Season Result:  2014-15
Season Result:  2013-14
Season Result:  2012-13
Season Result:  2011-12
Season Result:  2010-11
Season Result:  2009-10
Season Result:  2008-09
Season Result:  2007-08
Season Result:  2006-07


,date,away,home,away_score,home_score
0,2025-10-10,Grace Christian,Grace (IN),2,6
1,2025-10-10,Grace Christian,Grace (IN),1,15
2,2025-10-11,Grace (IN),Grace Christian,3,0
3,2025-10-11,Ohio Christian,Mount Vernon Nazarene (OH),9,5
4,2025-10-11,Ohio Christian,Mount Vernon Nazarene (OH),2,8
...,...,...,...,...,...
5005,2007-05-16,Mount Vernon Nazarene,Trinity International,3,6
5006,2007-05-17,St. Ambrose,Spring Arbor,3,11
5007,2007-05-17,Mount Vernon Nazarene,Trinity Christian,11,3
5008,2007-05-18,St. Ambrose,Spring Arbor,1,6


# Export dataframe as a csv file

In [9]:
df.to_csv('../data/raw/baseball_games.csv',index=False)

# Clean Dataset

In [19]:
# check for missing values
df_clean = pd.read_csv('../data/raw/baseball_games.csv')
missing_values = df_clean.isnull().sum()
#print(missing_values)

#df.info()

# change datatype of scores from object to numbers
df_clean = df_clean.astype({
    'away_score': int,
    'home_score': int
})


# standardize text data
df_clean['away'] = df_clean['away'].str.lower()
df_clean['home'] = df_clean['home'].str.lower()

same_team = {
    'grace' : 'grace (in)',
    'goshen' : 'goshen (in)',
    'huntington' : 'huntington (in)',
    'indiana wesleyan' : 'indiana wesleyan (in)',
    'marian' : 'marian (in)',
    'mount vernon nazarene' : 'mount vernon nazarene (oh)',
    'saint francis (ind.)' : 'saint francis (in)',
    'spring arbor' : 'spring arbor (in)',
    'taylor' : 'taylor (in)',
    'aquinas' : 'aquinas (mi)',
    'faulkner' : 'faulkner (al)',
    'bryan' : 'bryan (tn)',
    'union' : 'union (ky)'
}

df_clean['away'] = df_clean['away'].replace(same_team)
df_clean['home'] = df_clean['home'].replace(same_team)

In [20]:
# Create target vector (winning team)
df_clean['winning_team'] = np.where(df_clean['away_score'] > df_clean['home_score'], df_clean['away'], df_clean['home'])

In [21]:
df_clean = df_clean.sort_values(by='date', ascending=True)
df_clean.to_csv('../data/clean/baseball_games.csv', index=False)